# Viscosity solutions: regularized geodesic distances

The geodesic distance from a source set $\Gamma \subset M$ on a surface is the unique viscosity solution of the Eikonal equation:
$$ \| \nabla u(x) \| = 1 \quad \text{on } M \setminus \Gamma, \qquad u = 0 \text{ on } \Gamma. $$
Methods like fast marching solve this directly. The catch: the solution is only $C^0$. It has *creases* along the cut locus, which is the set of points reachable by two equally-short geodesics. These creases are intrinsic to the Eikonal but a nuisance for heat-based methods, gradient-based descriptors, or anything that wants $\nabla u$ to be smooth.

Edelstein, Guillen, Solomon, Ben-Chen (2023) replace the hard equality $\|\nabla u\| = 1$ with a *convex inequality* $\|\nabla u\| \le 1$ and add a smoothing regulariser:
$$ \max_u \int_M u \,\mathrm{d}A \;-\; \frac{\alpha}{2} R(u) \quad\text{s.t.}\quad \|\nabla u\|_{\text{face}} \le 1, \ u\big|_\Gamma = 0. $$
Without the $\alpha$ term this is exactly the problem solved in Belyaev & Fayolle (2020). 

The paper proposes three flavours of regulariser $R(u)$:

- **Dirichlet**: $R(u) = \int \|\nabla u\|^2$ smooths the gradient field globally.

- **Vector field alignment**: $R(u) = \int \|\nabla u\|^2 + \beta \langle \nabla u, vf \rangle^2$ adds a face-wise penalty on the component of $\nabla u$ along an input line field $vf$, which encourages the isolines of $u$ to *align with* $vf$.

- **Hessian**: $R(u) = \int \|H u\|^2$ penalises the second derivative.

All three are quadratic programs with one second-order-cone inequality per face, solved by the same ADMM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import igl
from src.rgd_admm import rgd_admm

## Setup

In [ ]:
mesh_name = 'spot_rr'
V, F = igl.read_triangle_mesh(f'../external/RGD/data/{mesh_name}.off')
x0 = 2272
print(f'Imported {mesh_name} with {V.shape[0]} vertices, {F.shape[0]} faces, source vertex = {x0}')

## No regularization and Dirichlet regularization

In [ ]:
alpha_no = 0.00
alpha_small = 0.05
alpha_large = 0.1

u_no, n0 = rgd_admm(V, F, x0, reg='D', alpha_hat=alpha_no)
u_small, n1 = rgd_admm(V, F, x0, reg='D', alpha_hat=alpha_small)
u_large, n2 = rgd_admm(V, F, x0, reg='D', alpha_hat=alpha_large)

for name, u, n in [(f'No regularization (alpha = {alpha_no:.2f})', u_no, n0),
                    (f'Dirichlet (alpha = {alpha_small:.2f})', u_small, n1),
                    (f'Dirichlet (alpha = {alpha_large:.2f})', u_large, n2)]:
    print(f'{name} {n} ADMM iters range [{u.min():.4f}, {u.max():.4f}]')

## Vector field alignment

We build a smooth tangent line field by projecting a constant 3D direction onto each face's tangent plane. Geodesics under this regulariser will route their isolines along the projected direction on the mesh surface.

In [ ]:
n_face = igl.per_face_normals(V, F.astype(np.int32), np.array([0.0, 0.0, 0.0]))
direction = np.array([0.0, 1.0, 0.0])
vf = direction[None, :] - (n_face @ direction)[:, None] * n_face
vf = vf / np.maximum(np.linalg.norm(vf, axis=1, keepdims=True), 1e-10)

u_vfa, n3 = rgd_admm(V, F, x0, reg='vfa', alpha_hat=0.05, beta_hat=100.0, vf=vf)
print(f'Vector field alignment (alpha=0.05, beta=100, +y vector field) {n3} iters '
      f'range [{u_vfa.min():.4f}, {u_vfa.max():.4f}]')

## Visualization

In [ ]:
import polyscope as ps

ps.init()
ps.remove_all_structures()
ps.set_ground_plane_mode('none')
ps.set_view_projection_mode('orthographic')
ps.load_color_map('RdBu_black', '../data/RdBu_black.png')

panels = [('No regularization (alpha=0)', u_no),
    ('Dirichlet alpha=0.05', u_small),
    ('Dirichlet alpha=0.15', u_large),
    ('vfa alpha=0.05 beta=100', u_vfa)]

vmin = float(min(u.min() for _, u in panels))
vmax = float(max(u.max() for _, u in panels))

# Plot meshes side-by-side
shift_step = 1.5 * (V[:, 0].max() - V[:, 0].min())
for k, (label, u) in enumerate(panels):
    Vk = V + np.array([k * shift_step, 0.0, 0.0])
    mesh = ps.register_surface_mesh(label, Vk, F, smooth_shade=True)
    mesh.add_scalar_quantity('u', u, cmap='RdBu_black', vminmax=(vmin, vmax), enabled=True)
    src_pt = Vk[x0:x0+1]
    ps.register_point_cloud(f'source ({label})', src_pt).set_color((1.0, 0.0, 0.0))

# Plot a subset of face vectors
vfa_idx = next(k for k, (label, _) in enumerate(panels) if label.startswith('vfa'))
Vk_vfa = V + np.array([vfa_idx * shift_step, 0.0, 0.0])
barycenters = (Vk_vfa[F[:, 0]] + Vk_vfa[F[:, 1]] + Vk_vfa[F[:, 2]]) / 3.0

target_n = min(1000, F.shape[0])
e1 = Vk_vfa[F[:, 1]] - Vk_vfa[F[:, 0]]
e2 = Vk_vfa[F[:, 2]] - Vk_vfa[F[:, 0]]
face_areas = 0.5 * np.linalg.norm(np.cross(e1, e2), axis=1)
p = face_areas / np.maximum(face_areas.sum(), 1e-12)
idx = np.random.default_rng(0).choice(F.shape[0], size=target_n, replace=False, p=p)

ps.register_point_cloud('vfa line field positions', barycenters[idx], radius=0.005, color=(255.0/255.0,255.0/255.0,153.0/255.0))\
    .add_vector_quantity('vf', vf[idx], color=(255.0/255.0,255.0/255.0,153.0/255.0), enabled=True, length=0.03, radius=0.005)

ps.show()

**▶ Your turn.** Try a much larger `alpha_hat` for the Dirichlet regularizer (say `0.5`). The isolines around the source stay almost identical, but the smoothing zones farther from the source widen. What happens to the number of ADMM iterations? Can you think of the reason? For vector field alignment, swap the alignment direction. Try `[1, 0, 0]` or `[0, 0, 1]` and watch the isolines re-orient to follow the new line field.

## References

[1] *A convex optimization framework for regularized geodesic distances*.
       Edelstein, M.; Guillen, N.; Solomon, J.; Ben-Chen, M.
       ACM Transactions on Graphics (SIGGRAPH) 42 (4): 1 (2023).
       :DOI:`10.1145/3592439`